In [9]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.signal import butter, filtfilt, iirnotch

In [10]:
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = max(lowcut / nyq, 0.001)
    high = min(highcut / nyq, 0.999)
    b, a = butter(order, [low, high], btype='band')
    return b, a

def butter_lowpass(cutoff, fs, order=4):
    nyq = 0.5 * fs
    cut = min(cutoff / nyq, 0.999)
    b, a = butter(order, cut, btype='low')
    return b, a


def preprocess_emg_signal(df, fs=1000):

    df_processed = df.copy()
    channels = [f'Channel {i}' for i in range(1, 4)]
    b_band, a_band = butter_bandpass(20, 450, fs, order=4)
    b_low, a_low = butter_lowpass(10, fs, order=4)
    b_notch, a_notch = iirnotch(w0=50, Q=30, fs=fs)

    for ch in channels:
        if ch not in df.columns:
            continue
        raw_sig = df[ch].fillna(0).values.astype(np.float32)
       
        sig_notch = filtfilt(b_notch, a_notch, raw_sig)
      
        filtered = filtfilt(b_band, a_band, sig_notch)
        
        rectified = np.abs(filtered)
      
        envelope = filtfilt(b_low, a_low, rectified)
        envelope = np.maximum(envelope, 0.0)
        df_processed[ch] = envelope

    return df_processed

In [11]:
def load_and_preprocess_dataset(base_dir="datasets/train", fs=1000):

    gesture_map = {
        '1st_Gesture_Resting': 0,
        '2nd_Gesture_Fist': 1,
        '4th_Gesture_Open_Hand': 2,
        '7th_Gesture_Pointing': 3
    }
    trials = []
    channels = ['Channel 1', 'Channel 2', 'Channel 3']
    
    for folder, cls_idx in gesture_map.items():
        files = sorted(glob.glob(os.path.join(base_dir, folder, "*.csv")))
        for f in files:
            df = pd.read_csv(f)
            df_proc = preprocess_emg_signal(df, fs=fs)
            trials.append({
                'file': os.path.basename(f),
                'signal': df_proc[channels].values,
                'class': cls_idx
            })
            
    print(f"Preprocessing finish: {len(trials)} trials ")
    return trials

trials = load_and_preprocess_dataset('datasets/train')

Preprocessing finish: 300 trials 


In [14]:
def extract_15_features(window_3ch, thresh=1e-5):
    features = []
    for ch in range(3):
        x = window_3ch[:, ch]
        xc = x - np.mean(x)
        diffs = np.diff(x)

        # MAV
        mav = np.mean(np.abs(x))
        # RMS
        rms = np.sqrt(np.mean(x ** 2))
        # WL
        wl = np.sum(np.abs(diffs))
        # ZC
        zc = np.sum(((xc[:-1] * xc[1:]) < 0) & (np.abs(diffs) >= thresh))
        # SSC
        diff1 = x[1:-1] - x[:-2]
        diff2 = x[1:-1] - x[2:]
        ssc = np.sum(((diff1 * diff2) > 0) & ((np.abs(diff1) >= thresh) | (np.abs(diff2) >= thresh)))

        features.extend([mav, rms, wl, float(zc), float(ssc)])
    return np.array(features, dtype=np.float32)

def create_sliding_windows(trials, window_size=200, step_size=25):
    X, y = [], []
    for t in trials:
        sig = t['signal']
        cls_val = t['class']
        for i in range(0, len(sig) - window_size + 1, step_size):
            window = sig[i : i + window_size]
            X.append(extract_15_features(window))
            y.append(cls_val)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

train_trials, temp_trials = train_test_split(trials, test_size=0.3, random_state=42, stratify=[t['class'] for t in trials])
val_trials, test_trials = train_test_split(temp_trials, test_size=0.5, random_state=42, stratify=[t['class'] for t in temp_trials])

print(f"Trials Number: Train={len(train_trials)}, Val={len(val_trials)}, Test={len(test_trials)}")

X_train, y_train = create_sliding_windows(train_trials, window_size=200, step_size=25)
X_val, y_val = create_sliding_windows(val_trials, window_size=200, step_size=25)
X_test, y_test = create_sliding_windows(test_trials, window_size=200, step_size=25)

mean = np.mean(X_train, axis=0, keepdims=True)
std = np.std(X_train, axis=0, keepdims=True) + 1e-8

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}   | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")


Trials Number: Train=210, Val=45, Test=45
X_train shape: (27930, 200, 3) | y_train shape: (27930,)
X_val shape:   (5985, 200, 3)   | y_val shape:   (5985,)
X_test shape:  (5985, 200, 3)  | y_test shape:  (5985,)


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(15,), batch_size=1),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(4, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()

with open('semg_mlp.tflite', 'wb') as f:
    f.write(tflite_model)
